<a href="https://colab.research.google.com/github/TorbenGl/visionseminar/blob/main/notebooks/04_vision_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning for Maritime Vision Applications
## Session 1 · Part 2 — Vision Transformers (ViT)
*Torben Globisch · VACOT*

---

| Notebook | Topic |
|---|---|
| 00 | Artificial Neural Networks: neurons, layers, learning |
| 01 | Computer Vision & an MLP on MNIST |
| 02 | Convolutional Neural Networks |
| 03 | Vision Foundation Models: the CNN era |
| **→ 04** | **Vision Transformers** |
| 05 | Self-supervised pretraining & transfer |

**By the end of this notebook you can**
1. explain self-attention with **queries, keys and values** and implement it in a few lines,
2. describe how a ViT **tokenises an image** (patches → embeddings → [CLS] + positional embeddings),
3. build and train a small ViT yourself (**Exercise**) and compare it with a CNN,
4. look inside a pretrained ViT: what do its positional embeddings and attention maps show?

> 💡 Use a GPU runtime: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
# @title ⚙️ Setup: imports, colours & helpers — run me first { display-mode: "form" }
import math, io, time, urllib.request
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle, FancyArrowPatch
from IPython.display import HTML, display
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

# Colour palette — the same colours as the lecture slides
NAVY, BLUE, LAVENDER, ORANGE, GREEN, RED, GREY, LIGHT = (
    "#2D3A8C", "#5B67B8", "#A3A9DB", "#E8702A", "#3E9E64", "#C0392B", "#8A8FA8", "#EEF0F7")
plt.rcParams.update({
    "figure.dpi": 100, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 11, "axes.titlecolor": NAVY,
    "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, GREEN, BLUE, RED, GREY]),
    "animation.html": "jshtml", "animation.embed_limit": 80,
})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0); np.random.seed(0)
print(f"PyTorch {torch.__version__} | device: {device}")
if device.type == "cpu":
    print("⚠️  No GPU found. In Colab: Runtime → Change runtime type → T4 GPU")

def play(anim):
    """Show a matplotlib animation as an HTML player (play / pause / step frame by frame)."""
    plt.close(anim._fig)
    return HTML(anim.to_jshtml())

# Maritime example photos from the COCO validation set (Flickr, Creative Commons licences).
MARITIME = {"trawler": 330369, "sailboat": 286422, "lifeboat": 400803, "marina": 442822,
            "fishing_boat": 451090, "yacht": 489014, "ferry": 543300}

def load_image(name, size=None):
    """Load one of the MARITIME photos (or any image URL) as an RGB PIL image."""
    url = f"http://images.cocodataset.org/val2017/{MARITIME[name]:012d}.jpg" if name in MARITIME else name
    with urllib.request.urlopen(url, timeout=30) as r:
        img = Image.open(io.BytesIO(r.read())).convert("RGB")
    if size is not None:  # centre-crop to a square, then resize
        s = min(img.size); l, t = (img.width - s) // 2, (img.height - s) // 2
        img = img.crop((l, t, l + s, t + s)).resize((size, size), Image.BICUBIC)
    return img

---
# 1 · Transformers: from sequences to context

*Vaswani et al., 2017: "Attention Is All You Need"*, originally for machine translation (text in, text out).

**Core idea:** the input is a **sequence of tokens** (words, pixels, image patches, …) and **every token can look at every
other token**, all at the same time.

**Why does context matter?** The word *"bank"* means different things in
* "I went to the **bank** to deposit money." (finance)
* "We sat on the river **bank**." (geography)

A token has to look at its neighbours to understand its own meaning. The Transformer is the backbone of all modern
large language models (GPT, BERT, Claude, …). In 2021 it was applied to images for the first time.

## 1.1 · Self-attention: queries, keys and values
Every token embedding $x$ produces three vectors via learned linear maps:

| | | |
|---|---|---|
| **Q**uery | $q = W_Q\,x$ | "What am I looking for?" |
| **K**ey | $k = W_K\,x$ | "What can I offer to others?" |
| **V**alue | $v = W_V\,x$ | "What I actually send if I'm chosen" |

$$ \text{Attention}(Q, K, V) = \text{softmax}\Big(\frac{QK^\top}{\sqrt{d}}\Big)\,V $$

* $QK^\top$: how well each query matches each key (dot products)
* $\sqrt{d}$: scaling that keeps the softmax (and the gradients) well-behaved
* softmax: turns each row of scores into weights that sum to 1
* $\cdot V$: every output is a **weighted mix** of all values

It really is only a few lines of code:

In [ ]:
def self_attention(X, W_q, W_k, W_v):
    """X: (n_tokens, d_model). Returns the outputs (n_tokens, d) and the attention weights (n_tokens, n_tokens)."""
    Q, K, V = X @ W_q, X @ W_k, X @ W_v
    scores = Q @ K.T / math.sqrt(K.shape[-1])     # (n, n): query i vs. key j
    A = scores.softmax(dim=-1)                    # each row sums to 1
    return A @ V, A

### A toy example: *"I love cats and dogs"*
We give each word a hand-made 4-D embedding with the features *[animal, verb, pronoun, conjunction]*. With a simple
choice of $W_Q, W_K$ ("animals look for animals") we can predict who attends to whom. In a real Transformer these
matrices are *learned*.

In [ ]:
tokens = ["I", "love", "cats", "and", "dogs"]
X = torch.tensor([[0.0, 0.0, 1.0, 0.0],     # I     → pronoun
                  [0.0, 1.0, 0.3, 0.0],     # love  → verb (a bit pronoun-related: who loves?)
                  [1.0, 0.2, 0.0, 0.0],     # cats  → animal
                  [0.0, 0.0, 0.0, 1.0],     # and   → conjunction
                  [1.0, 0.1, 0.0, 0.2]])    # dogs  → animal
W_q = torch.diag(torch.tensor([3.0, 1.5, 1.5, 0.5]))   # queries ask mostly for "animal"-ness
W_k = torch.eye(4) * 1.5
W_v = torch.eye(4)
out, A = self_attention(X, W_q, W_k, W_v)

# Sanity check against PyTorch's built-in (fused, fast) implementation
ref = F.scaled_dot_product_attention((X @ W_q)[None], (X @ W_k)[None], (X @ W_v)[None])[0]
print("matches F.scaled_dot_product_attention:", torch.allclose(out, ref, atol=1e-6))

In [ ]:
# @title 🎞️ Animation: who attends to whom? { display-mode: "form" }
fig, (ax_m, ax_a) = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw=dict(width_ratios=[1, 1.5]))
ax_m.imshow(A, cmap="Blues", vmin=0, vmax=1)
for (i, j), v in np.ndenumerate(A.numpy()): ax_m.text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if v > .5 else NAVY)
ax_m.set_xticks(range(5), tokens); ax_m.set_yticks(range(5), tokens); ax_m.set_xlabel("key (attended to)"); ax_m.set_ylabel("query (attending)")
ax_m.set_title("attention weights softmax(QKᵀ/√d)")
row_box = Rectangle((-.5, -.5), 5, 1, fill=False, ec=ORANGE, lw=3); ax_m.add_patch(row_box)
ax_a.axis("off"); ax_a.set_xlim(-.5, 4.5); ax_a.set_ylim(-0.6, 1.6)
for j, t in enumerate(tokens):
    ax_a.text(j, 1.3, t, ha="center", va="center", fontsize=14, bbox=dict(boxstyle="round", fc=LIGHT, ec=NAVY))
    ax_a.text(j, -0.3, t + "'", ha="center", va="center", fontsize=14, bbox=dict(boxstyle="round", fc=LAVENDER, ec=NAVY))
ax_a.text(-0.5, 1.55, "input tokens", color=GREY); ax_a.text(-0.5, -0.6, "context-aware output tokens", color=GREY)
arrows = [ax_a.annotate("", xy=(0, -0.05), xytext=(0, 1.05), arrowprops=dict(arrowstyle="-|>", color=ORANGE, lw=1)) for _ in range(5)]
title = ax_a.set_title("")
plt.tight_layout()

def update(i):
    row_box.set_xy((-.5, i - .5))
    for j, arr in enumerate(arrows):
        arr.xy = (i, -0.05); arr.set_position((j, 1.05))
        arr.arrow_patch.set_linewidth(0.5 + 12 * A[i, j].item()); arr.arrow_patch.set_alpha(0.25 + 0.75 * A[i, j].item())
    title.set_text(f"output token '{tokens[i]}′' = weighted mix of all value vectors (arrow width = weight)")
    return arrows + [row_box]

play(animation.FuncAnimation(fig, update, frames=5, interval=1500))

"cats" attends most strongly to "dogs" (and to itself): both are animals. **vs. CNN:** a 3 × 3 kernel only sees its
direct neighbours, while self-attention sees the **entire sequence from the very first layer**.

In practice a Transformer runs several attention computations in parallel (**multi-head attention**), each with its own
$W_Q, W_K, W_V$, so different heads can look for different relationships.

---
# 2 · Vision Transformer, step 1: image → patches
*Dosovitskiy et al., 2021 (Google): "An Image is Worth 16x16 Words"*

**Key insight: treat an image like a sentence**, with patches as words.
* choose a patch size $p$ (e.g. $p = 16$ pixels)
* cut the image into non-overlapping $p \times p$ patches; this is a convolution with **kernel size = stride = p** (notebook 03!)
* a 224 × 224 image with $p = 16$ gives $\frac{224}{16}\times\frac{224}{16} = 14 \times 14 = 196$ patches

In [ ]:
img = load_image("trawler", size=224)
x_img = torch.tensor(np.asarray(img), dtype=torch.float32).permute(2, 0, 1) / 255     # (3, 224, 224)
p = 16
patches = x_img.unfold(1, p, p).unfold(2, p, p)                    # (3, 14, 14, 16, 16)
patches = patches.permute(1, 2, 0, 3, 4).reshape(-1, 3, p, p)      # (196, 3, 16, 16): row by row
print("patches:", tuple(patches.shape))

In [ ]:
# @title 🎞️ Animation: cutting the image into a sequence of patches { display-mode: "form" }
fig = plt.figure(figsize=(15, 5.6))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.9])
ax_i = fig.add_subplot(gs[0]); ax_s = fig.add_subplot(gs[1])
ax_i.imshow(img); ax_i.axis("off"); ax_i.set_title("input image, 16 × 16 patches")
for k in range(0, 225, 16): ax_i.axhline(k - .5, color="white", lw=.6); ax_i.axvline(k - .5, color="white", lw=.6)
row_rect = Rectangle((-.5, -.5), 224, 16, fill=True, fc=ORANGE, alpha=.35, ec=ORANGE, lw=2.5); ax_i.add_patch(row_rect)
PER_LINE, STEP = 28, 18                                         # 28 tokens per line → 7 lines of the sequence
ax_s.axis("off"); ax_s.set_xlim(-40, PER_LINE * STEP + 2); ax_s.set_ylim(-7 * 30 - 4, 16)
ax_s.set_title("…flattened into one long sequence of tokens  P1, P2, …, P196")
strip_imgs = []
for k in range(196):
    r, c = divmod(k, PER_LINE)
    strip_imgs.append(ax_s.imshow(patches[k].permute(1, 2, 0), extent=[c * STEP, c * STEP + 16, -r * 30 - 16, -r * 30], visible=False))
for r in range(7): ax_s.text(-6, -r * 30 - 8, f"P{r * PER_LINE + 1}", ha="right", va="center", fontsize=8, color=GREY)
counter = ax_s.text(PER_LINE * STEP, 10, "", ha="right", color=NAVY, fontweight="bold")
plt.tight_layout()

def update(t):
    r = min(t, 13)
    row_rect.set_xy((-.5, r * 16 - .5)); row_rect.set_visible(t < 14)
    for k in range(196): strip_imgs[k].set_visible(k < (r + 1) * 14)
    counter.set_text(f"{(r + 1) * 14} / 196 tokens")
    return strip_imgs + [row_rect]

play(animation.FuncAnimation(fig, update, frames=16, interval=400))

**Connection to stride.** Flattening each patch and multiplying with a weight matrix is *exactly* a convolution with
kernel size $p$ and stride $p$. The ViT "patch embedding layer" literally *is* a conv layer. Let's prove it:

In [ ]:
d_model = 768
conv = nn.Conv2d(3, d_model, kernel_size=p, stride=p)               # the usual ViT implementation
linear = nn.Linear(3 * p * p, d_model)                             # "flatten each patch, then linear projection"
linear.weight.data = conv.weight.data.reshape(d_model, -1)         # same parameters, just reshaped
linear.bias.data = conv.bias.data

tokens_conv = conv(x_img[None]).flatten(2).transpose(1, 2)[0]      # (196, 768)
tokens_linear = linear(patches.reshape(196, -1))                   # (196, 768)
print("conv output:", tuple(tokens_conv.shape), "| linear output:", tuple(tokens_linear.shape))
print("identical:", torch.allclose(tokens_conv, tokens_linear, atol=1e-4))

---
# 3 · Step 2: patch embeddings, [CLS] token and positional embeddings
1. **Flatten** each patch: a $p\times p$ patch with $C$ channels gives $p^2C$ values ($16\cdot16\cdot3 = 768$)
2. **Linear projection** to a $d$-dimensional token embedding (ViT-Base: $d = 768$)
3. **Prepend a learnable [CLS] token**: it has no patch of its own; it will collect a summary of the whole image
4. **Add positional embeddings**: attention itself has no notion of order. Without them, shuffling the patches would
   not change the output, just like the MLP in notebook 01!

In [ ]:
B, N, d = 1, 196, 768
patch_tokens = tokens_conv[None]                                   # (B, 196, 768)
cls_token = nn.Parameter(torch.zeros(1, 1, d))                     # learnable
pos_embedding = nn.Parameter(torch.randn(1, N + 1, d) * .02)       # learnable, one per position

seq = torch.cat([cls_token.expand(B, -1, -1), patch_tokens], dim=1)   # (B, 197, 768)
seq = seq + pos_embedding
print("token sequence fed to the Transformer encoder:", tuple(seq.shape), "= [CLS] + 196 patch tokens")

### What do learned positional embeddings look like?
We load a real ViT-Base/16 (pretrained on ImageNet-21k, fine-tuned on ImageNet-1k) and compare the positional embedding
of every patch position with all other positions (cosine similarity). Each small square shows one position's
similarity to all 14 × 14 positions:

In [ ]:
from transformers import ViTForImageClassification, ViTImageProcessor
vit = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224", attn_implementation="eager").eval().to(device)
vit_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

pos = vit.vit.embeddings.position_embeddings[0, 1:].detach().cpu()        # (196, 768), skip [CLS]
sim = F.normalize(pos, dim=1) @ F.normalize(pos, dim=1).T                 # (196, 196)

fig, axes = plt.subplots(14, 14, figsize=(8, 8))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(sim[k].reshape(14, 14), cmap="viridis", vmin=-.2, vmax=1); ax.set_xticks([]); ax.set_yticks([])
fig.supxlabel("input patch column", color=NAVY); fig.supylabel("input patch row", color=NAVY)
plt.suptitle("Similarity of learned positional embeddings (ViT-B/16)", color=NAVY, fontweight="bold")
plt.tight_layout(); plt.show()

Nobody told the model about 2-D geometry, yet each position is most similar to **itself and its neighbours**, and to
positions in the same row and column. The network **learned** the image grid from data.

---
# 4 · Step 3: Transformer encoder & classification
The encoder is a stack of $L$ identical blocks (ViT-Base: $L = 12$). Each block:

$$ x \leftarrow x + \text{MultiHeadAttention}(\text{LayerNorm}(x)) \qquad\qquad x \leftarrow x + \text{MLP}(\text{LayerNorm}(x)) $$

Note the **"+ x"**: these are the residual connections from ResNet (notebook 03)!

**Classification via [CLS]:** after the encoder, the [CLS] output has gathered information from **all** patches through
attention. A small linear head maps it to class scores; the patch tokens are discarded for classification.

In [ ]:
# @title 🎨 Figure: the complete ViT pipeline { display-mode: "form" }
fig, ax = plt.subplots(figsize=(15, 4.6)); ax.axis("off"); ax.set_xlim(0, 15); ax.set_ylim(0, 5)
small = img.resize((64, 64))
ax.imshow(small, extent=[0.1, 2.1, 1.5, 3.5])
for k in np.linspace(0.1, 2.1, 5): ax.plot([k, k], [1.5, 3.5], color="white", lw=1); ax.plot([0.1, 2.1], [k + 1.4, k + 1.4], color="white", lw=1)
ax.text(1.1, 1.2, "image → patches", ha="center", color=NAVY)
ax.annotate("", xy=(3.0, 2.5), xytext=(2.3, 2.5), arrowprops=dict(arrowstyle="-|>", color=GREY, lw=2))
cols = [ORANGE] + [LAVENDER] * 6
for i, c in enumerate(cols):
    ax.add_patch(FancyBboxPatch((3.2 + i * 0.55, 2.1), 0.42, 0.8, boxstyle="round,pad=0.02", fc=c, ec=NAVY))
    ax.text(3.41 + i * 0.55, 2.5, "CLS" if i == 0 else f"P{i}", ha="center", va="center", fontsize=7, rotation=90 if i == 0 else 0)
    ax.text(3.41 + i * 0.55, 3.15, "⊕", ha="center", va="center", fontsize=12, color=GREEN)
ax.text(5.1, 3.55, "+ positional embeddings", ha="center", color=GREEN, fontsize=9)
ax.text(5.1, 1.75, "linear projection + [CLS]", ha="center", color=NAVY, fontsize=9)
ax.annotate("", xy=(7.4, 2.5), xytext=(7.0, 2.5), arrowprops=dict(arrowstyle="-|>", color=GREY, lw=2))
ax.add_patch(FancyBboxPatch((7.5, 1.0), 3.1, 3.2, boxstyle="round,pad=0.05", fc=NAVY, ec="none"))
ax.text(9.05, 3.75, "Transformer encoder × L", ha="center", color="white", fontweight="bold")
for j, (lbl, y) in enumerate([("LayerNorm → Multi-Head Attention", 2.9), ("+ residual", 2.4), ("LayerNorm → MLP", 1.9), ("+ residual", 1.4)]):
    ax.text(9.05, y, lbl, ha="center", color="white" if "residual" not in lbl else LAVENDER, fontsize=9)
ax.annotate("", xy=(11.1, 3.3), xytext=(10.7, 3.3), arrowprops=dict(arrowstyle="-|>", color=GREY, lw=2))
ax.add_patch(FancyBboxPatch((11.2, 2.9), 0.42, 0.8, boxstyle="round,pad=0.02", fc=ORANGE, ec=NAVY)); ax.text(11.41, 3.3, "CLS", rotation=90, ha="center", va="center", fontsize=7)
ax.text(11.41, 2.6, "only [CLS] is used", ha="center", fontsize=8, color=GREY)
ax.annotate("", xy=(12.3, 3.3), xytext=(11.75, 3.3), arrowprops=dict(arrowstyle="-|>", color=GREY, lw=2))
ax.add_patch(FancyBboxPatch((12.4, 2.9), 1.5, 0.8, boxstyle="round,pad=0.05", fc=GREEN, ec="none")); ax.text(13.15, 3.3, "linear head", ha="center", va="center", color="white", fontweight="bold")
ax.text(14.1, 3.3, '"fireboat"', va="center", fontsize=10, color=NAVY)
plt.show()

---
# ✏️ Exercise: build a mini-ViT for MNIST
A real ViT is too big to train in a seminar, but the recipe works at any scale. MNIST images are 28 × 28; with a patch size
of 7 we get $4 \times 4 = 16$ tokens.

Complete `forward` below, following the three steps. PyTorch provides the encoder blocks (`nn.TransformerEncoderLayer`),
so you only have to do the *vision-specific* part.

In [ ]:
class MiniViT(nn.Module):
    def __init__(self, img_size=28, patch=7, in_ch=1, d=64, depth=4, heads=4, n_classes=10):
        super().__init__()
        n_patches = (img_size // patch) ** 2
        self.patch_embed = nn.Conv2d(in_ch, d, kernel_size=patch, stride=patch)   # step 1: patches → tokens
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d))                       # step 2: [CLS] …
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, d) * .02)    #         … + positions
        layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=4 * d, dropout=0.0, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)                      # step 3: encoder
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, n_classes)                                       #         classification head

    def forward(self, x):
        if x.dim() == 3: x = x.unsqueeze(1)          # (B, 28, 28) → (B, 1, 28, 28)
        # TODO 1: x = patch embedding, then flatten the 4×4 grid into a sequence: (B, d, 4, 4) → (B, 16, d)
        # TODO 2: prepend the [CLS] token (expand it to the batch size) → (B, 17, d), then add self.pos_embed
        # TODO 3: run self.encoder, apply self.norm, and feed ONLY the [CLS] output (position 0) to self.head
        raise NotImplementedError

In [ ]:
# @title 💡 Solution { display-mode: "form" }
class MiniViT(MiniViT):
    def forward(self, x):
        if x.dim() == 3: x = x.unsqueeze(1)
        x = self.patch_embed(x).flatten(2).transpose(1, 2)              # (B, 16, d)
        cls = self.cls_token.expand(len(x), -1, -1)                     # (B, 1, d)
        x = torch.cat([cls, x], dim=1) + self.pos_embed                 # (B, 17, d)
        x = self.norm(self.encoder(x))
        return self.head(x[:, 0])                                       # classify from [CLS]

In [ ]:
# @title Data & training helpers (same as notebooks 01/02) { display-mode: "form" }
from torchvision import datasets
from torch.utils.data import TensorDataset, DataLoader
mnist_train = datasets.MNIST("data", train=True, download=True); mnist_test = datasets.MNIST("data", train=False, download=True)
X_train = ((mnist_train.data.float() / 255) - 0.1307) / 0.3081; X_test = ((mnist_test.data.float() / 255) - 0.1307) / 0.3081
y_train, y_test = mnist_train.targets, mnist_test.targets
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=1000)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct, n = 0, 0
    for x, y in loader:
        correct += (model(x.to(device)).argmax(1) == y.to(device)).sum().item(); n += len(y)
    return correct / n

def fit(model, epochs=5, lr=1e-3):
    model.to(device); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_loader))
    accs = []
    for epoch in range(epochs):
        model.train(); t0 = time.time()
        for x, y in train_loader:
            loss = F.cross_entropy(model(x.to(device)), y.to(device))
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        accs.append(evaluate(model, test_loader))
        print(f"epoch {epoch + 1}/{epochs}  test acc {accs[-1]:.2%}  ({time.time() - t0:.1f}s)")
    return accs

count_params = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
torch.manual_seed(0)
mini_vit = MiniViT()
print(f"MiniViT parameters: {count_params(mini_vit):,}")
acc_vit = fit(mini_vit, epochs=5, lr=2e-3)

For comparison: the `SimpleCNN` from notebook 02 reached about **99 %** with 108 k parameters in the same 5 epochs.
Our mini-ViT gets close, but it is usually a little behind. Why?

The CNN has **inductive biases** built in (locality, weight sharing, translation equivariance); the ViT must
*learn* them from data, as we saw with the positional embeddings. With little data, the CNN wins. With a *lot* of data
(ViT was pretrained on 14 M to 300 M images) the ViT catches up and overtakes, because it is more flexible.

Let's look at what our mini-ViT's [CLS] token attends to in the last layer:

In [ ]:
@torch.no_grad()
def cls_attention_last_layer(model, x):
    """Re-run the last encoder layer's attention manually to read out the [CLS] → patch weights."""
    model.eval(); x = x.to(device).unsqueeze(1)
    h = model.patch_embed(x).flatten(2).transpose(1, 2)
    h = torch.cat([model.cls_token.expand(len(h), -1, -1), h], 1) + model.pos_embed
    for layer in model.encoder.layers[:-1]: h = layer(h)
    last = model.encoder.layers[-1]
    _, w = last.self_attn(last.norm1(h), last.norm1(h), last.norm1(h), need_weights=True, average_attn_weights=True)
    return w[:, 0, 1:].reshape(-1, 4, 4).cpu()                      # [CLS] row, patch columns

att = cls_attention_last_layer(mini_vit, X_test[:8])
fig, axes = plt.subplots(2, 8, figsize=(14, 3.8))
for k in range(8):
    axes[0, k].imshow(mnist_test.data[k], cmap="gray_r"); axes[0, k].axis("off")
    axes[1, k].imshow(mnist_test.data[k], cmap="gray_r")
    axes[1, k].imshow(att[k], cmap="magma", alpha=.6, extent=[-.5, 27.5, 27.5, -.5]); axes[1, k].axis("off")
axes[0, 0].set_title("input", loc="left"); axes[1, 0].set_title("[CLS] attention (4×4 patches)", loc="left")
plt.tight_layout(); plt.show()

---
# 5 · A pretrained ViT on maritime images
Back to the real ViT-Base/16. It classifies our boats, and we can watch its [CLS] token's attention change through the 12 layers.

In [ ]:
names = ["trawler", "sailboat", "lifeboat", "ferry"]
boat_imgs = [load_image(n, size=224) for n in names]
with torch.no_grad():
    inputs = vit_processor(images=boat_imgs, return_tensors="pt").to(device)
    outputs = vit(**inputs, output_attentions=True)
probs = outputs.logits.softmax(-1).cpu()

fig, axes = plt.subplots(1, 4, figsize=(15, 4.2))
for ax, im, pr in zip(axes, boat_imgs, probs):
    ax.imshow(im); ax.axis("off")
    top = pr.topk(3)
    ax.set_title("\n".join(f"{vit.config.id2label[i.item()].split(',')[0]}: {v:.0%}" for v, i in zip(top.values, top.indices)), fontsize=9.5)
plt.tight_layout(); plt.show()

In [ ]:
# @title 🎞️ Animation: [CLS] attention through the 12 layers { display-mode: "form" }
# attentions: tuple of 12 tensors, each (batch, heads, 197, 197)
cls_maps = torch.stack([a[:, :, 0, 1:].mean(1) for a in outputs.attentions], 1).cpu()     # (batch, 12, 196)
cls_maps = cls_maps.reshape(len(names), 12, 14, 14)
fig, axes = plt.subplots(1, 4, figsize=(15, 4.3))
overlays = []
for ax, im in zip(axes, boat_imgs):
    ax.imshow(im); ax.axis("off")
    overlays.append(ax.imshow(np.zeros((14, 14)), cmap="inferno", alpha=.65, extent=[0, 224, 224, 0], vmin=0, vmax=1))
title = fig.suptitle("", color=NAVY, fontweight="bold"); plt.tight_layout()

def update(l):
    for k, ov in enumerate(overlays):
        m = cls_maps[k, l]; ov.set_data((m - m.min()) / (m.max() - m.min() + 1e-8))
    title.set_text(f"layer {l + 1} / 12: where does the [CLS] token look? (mean over heads)")
    return overlays

play(animation.FuncAnimation(fig, update, frames=12, interval=700))

### 🔍 Wait, why is it looking at the sky?
In the first layers the [CLS] attention roughly follows the image content. But from the middle layers on, it locks onto a
handful of **background patches** (sky, water, a corner), not the boat! This is not a bug in our code. It is a real
phenomenon that researchers only explained in 2023 (Darcet et al., *"Vision Transformers Need Registers"*): large ViTs
re-purpose a few uninformative background patches as internal **scratch-pad memory** for global information. Those
tokens have an unusually large norm. Let's check that ourselves:

In [ ]:
with torch.no_grad():
    hidden = vit(**inputs, output_hidden_states=True).hidden_states       # 13 tensors (embeddings + 12 layers), each (B, 197, 768)
LAYER = 7
token_norm = hidden[LAYER][:, 1:].norm(dim=-1).cpu().reshape(-1, 14, 14)   # (B, 14, 14)
att_l = cls_maps[:, LAYER - 1]

fig, axes = plt.subplots(2, 4, figsize=(15, 7.4))
for k, im in enumerate(boat_imgs):
    for r, (m, cmap, lbl) in enumerate([(att_l[k], "inferno", f"[CLS] attention, layer {LAYER}"), (token_norm[k], "viridis", f"token norm ‖x‖, layer {LAYER}")]):
        axes[r, k].imshow(im); axes[r, k].imshow(m, cmap=cmap, alpha=.7, extent=[0, 224, 224, 0]); axes[r, k].axis("off")
        axes[r, k].set_title(lbl if k == 0 else "", loc="left")
    top_att = set(att_l[k].flatten().topk(5).indices.tolist()); top_norm = set(token_norm[k].flatten().topk(5).indices.tolist())
    ratio = (token_norm[k].max() / token_norm[k].median()).item()
    axes[1, k].text(4, 220, f"top-5 overlap: {len(top_att & top_norm)}/5\nmax/median norm: {ratio:.0f}×", color="white", fontsize=9,
                    bbox=dict(fc=NAVY, alpha=.8, ec="none"))
plt.tight_layout(); plt.show()

The patches that receive most attention are exactly the **high-norm "artifact" tokens**. Two lessons:
1. **Attention maps are not explanations.** Where a model *attends* is not necessarily what it *uses* for its decision.
2. This is an active research area, and the fix is simple: add a few extra learnable tokens ("registers") next to [CLS] that the
   model can use as memory instead. Newer models (e.g. DINOv2 *with registers*) do exactly that.

In notebook 05 we will see that a *self-supervised* ViT (DINO) gives remarkably clean object attention.

---
# 6 · CNN vs. ViT

| | CNN | ViT |
|---|---|---|
| basic operation | convolution (local, shared kernel) | self-attention (global, data-dependent weights) |
| receptive field in layer 1 | 3 × 3 pixels | the whole image |
| built-in assumptions (inductive bias) | locality, translation equivariance | almost none (positions are learned) |
| data needed | works well on small/medium datasets | needs large-scale pretraining |
| scales with data & compute | good | excellent: the basis of most modern VFMs |

---
# Summary
1. **Self-attention:** every token builds a query, key and value; outputs are $\text{softmax}(QK^\top/\sqrt d)\,V$, i.e. a weighted mix of all tokens.
2. **ViT step 1:** cut the image into $p\times p$ patches; this is a conv with kernel = stride = $p$ (224 px, $p=16$ → 196 tokens).
3. **ViT step 2:** linear projection + learnable **[CLS]** token + learnable **positional embeddings** (which learn the 2-D grid by themselves).
4. **ViT step 3:** a stack of Transformer blocks (attention + MLP, with residuals); classify from the [CLS] output.
5. ViTs have **less inductive bias** than CNNs: worse on small data, better at scale. So how do we pretrain them at scale
   **without** millions of labels? → **Next:** [05 · Self-supervised pretraining & transfer](https://colab.research.google.com/github/TorbenGl/visionseminar/blob/main/notebooks/05_self_supervised_learning_and_transfer.ipynb)